In [ ]:
# Notebook version: 1.0.0 (DSPCL; see README for change history)
from __future__ import print_function
from __future__ import absolute_import
from __future__ import unicode_literals

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
import math as m
from time import time

from scipy import special
import pandas as pd

from IPython.display import HTML, display
import ipywidgets as widgets

%matplotlib inline

deg2rad = lambda deg: deg / 180. * np.pi
rad2deg = lambda rad: rad * 180. / np.pi

import warnings
warnings.filterwarnings('ignore')

# --- module-specific imports go here (add a commented pip/conda line above each; standard stack only) ---


# --- DSP_COLORS palette (DSPCL S.2.2): POTE1_COLORS unchanged + DSP extensions ---
DSP_COLORS = {
    # ===== Inherited from POTE1_COLORS (STYLE_GUIDE_V3_R3 S.2.2) - canonical, do not drift =====
    # --- Solar system / space (Module 2) ---
    'sun':              '#FFD700',   # Gold
    'terrestrial':      '#c1440e',   # Mars red
    'giant':            '#c88b3a',   # Jupiter tan
    'space_bg':         '#0d1b2a',   # Deep space dark blue
    # --- Earth structural layers (Modules 3, 4, 5) ---
    'inner_core':       '#b5121b',   # Deep red
    'outer_core':       '#e8401c',   # Orange-red
    'lower_mantle':     '#e07b39',   # Burnt orange
    'upper_mantle':     '#e8a838',   # Amber
    'asthenosphere':    '#c8b870',   # Warm tan (partial melt)
    'oceanic_crust':    '#2e6fa3',   # Ocean blue
    'continental_crust':'#a08050',   # Sandy brown
    'water':            '#aad4f5',   # Pale blue
    # --- Plate tectonics (Module 3) ---
    'divergent':        '#d62728',   # Red
    'convergent':       '#1f77b4',   # Blue
    'transform':        '#2ca02c',   # Green
    'hotspot':          '#ff7f0e',   # Orange
    'slab':             '#17becf',   # Cyan (cold slab)
    # --- Geophysical fields (Modules 4–10) ---
    'gravity_pos':      '#d62728',   # Positive anomaly red
    'gravity_neg':      '#1f77b4',   # Negative anomaly blue
    'mag_normal':       '#2c2c2c',   # Normal polarity black
    'mag_reversed':     '#f0f0f0',   # Reversed polarity white/light
    'heat_high':        '#d62728',   # High heat flow red
    'heat_low':         '#1f77b4',   # Low heat flow blue
    # --- General purpose ---
    'highlight':        '#ff7f0e',   # Orange accent
    'neutral':          '#7f7f7f',   # Medium gray
    'earth':            '#4fa8d5',   # Earth blue (general Earth reference)
    'accent':           '#ffffff',   # White accent
    # ===== DSP extensions =====
    # --- Domains ---
    'time_domain':      '#1f77b4',   # Blue: signals plotted against time/space
    'freq_domain':      '#d62728',   # Red: spectra plotted against frequency/wavenumber
    'phase':            '#9467bd',   # Purple: phase spectra
    # --- Continuous vs sampled ---
    'continuous':       '#7f7f7f',   # Gray: underlying analog signal
    'sampled':          '#1f77b4',   # Blue: discrete samples (stems/markers)
    'alias':            '#ff7f0e',   # Orange: aliased components / spectral replicas
    # --- Systems ---
    'input':            '#1f77b4',   # Blue: system input x
    'impulse_resp':     '#2ca02c',   # Green: impulse response h
    'output':           '#d62728',   # Red: system output y
    'noise':            '#bcbd22',   # Olive: additive noise
    # --- Windows, filters, Z-plane ---
    'window':           '#8c564b',   # Brown: window/taper functions
    'passband':         '#aad4f5',   # Pale blue: filter passband shading
    'pole':             '#d62728',   # Red: poles (x markers)
    'zero':             '#1f77b4',   # Blue: zeros (o markers)
    'unit_circle':      '#2c2c2c',   # Near-black: |z| = 1
}

# --- rcParams defaults (S.2.1) ---
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'sans-serif',
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.labelsize':   12,
    'xtick.labelsize':  11,
    'ytick.labelsize':  11,
    'legend.fontsize':  11,
    'figure.dpi':       120,
    'lines.linewidth':  2.0,
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

# --- Animation embed size limit (MB) — increase if animations are large ---
plt.rcParams['animation.embed_limit'] = 50

# --- Full-width notebook display ---
display(HTML("<style>.container { width:100% !important; }</style>"))

# --- Robust figure/table centering (classic Notebook 6 + JupyterLab) ---
display(HTML("""
<style>
.output_png { text-align: center; }
.output_area .output_png img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output table { margin-left: auto; margin-right: auto; }
.output_area table { margin-left: auto; margin-right: auto; }
/* center every table: Markdown tables and pandas/HTML tables in outputs (DSPCL ruling 2026-10-05) */
.jp-RenderedHTMLCommon table, .rendered_html table, .jp-RenderedMarkdown table {
    margin-left: auto !important; margin-right: auto !important; }
</style>
"""))

# --- Code-cell toggle (hidden by default for cleaner reading) ---
display(HTML('''<style>
.code-toggle-btn {
    background: #2c3e50; color: white; border: none;
    padding: 5px 12px; border-radius: 4px; cursor: pointer;
    font-size: 13px; margin: 4px 0;
}
.code-toggle-btn:hover { background: #34495e; }
</style>
<script>
var pote1_code_shown = true;
function pote1ToggleCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    var nodes = document.querySelectorAll(sel);
    pote1_code_shown = !pote1_code_shown;
    nodes.forEach(function(el){ el.style.display = pote1_code_shown ? '' : 'none'; });
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = pote1_code_shown ? 'Hide code' : 'Show code';
}
function pote1HideCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    document.querySelectorAll(sel).forEach(function(el){ el.style.display='none'; });
    pote1_code_shown = false;
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = 'Show code';
}
/* Hide on load; retry briefly while the notebook DOM finishes rendering */
setTimeout(pote1HideCode, 300);
setTimeout(pote1HideCode, 1200);
</script>
<button class="code-toggle-btn" onclick="pote1ToggleCode()">
  <span id="pote1-toggle-label">Show code</span>
</button>
<span style="font-size:12px; color:#555; margin-left:8px;">
  Only the Python code is hidden — text and figures stay visible.
</span>'''))

In [ ]:
# Course title banner (Python-generated): a seismic gather, a sampled waveform, and the gather's computed spectrum
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap

_C = DSP_COLORS

def _ricker(t, f0):
    a = (np.pi * f0 * t) ** 2
    return (1 - 2 * a) * np.exp(-a)

def glow_text(ax, x, y, txt, fs):
    ax.text(x, y, txt, ha='center', va='center', fontsize=fs, color='white', fontweight='bold', zorder=20,
            path_effects=[pe.withStroke(linewidth=3.5, foreground='#0a1830'), pe.Normal()])

# --- synthetic shot gather: three reflections with hyperbolic moveout, Ricker source wavelet ---
_dt, _nt, _ntr, _f0 = 0.002, 400, 12, 30.0          # 2 ms sampling, 0.8 s, 12 traces, 30 Hz Ricker
_t = np.arange(_nt) * _dt
_off = np.linspace(0, 800, _ntr)                     # offsets (m)
_gather = np.zeros((_ntr, _nt))
for _t0, _v, _amp in [(0.14, 1900, 1.0), (0.34, 2300, -0.75), (0.56, 2700, 0.6)]:
    _tx = np.sqrt(_t0 ** 2 + (_off / _v) ** 2)
    _gather += _amp * _ricker(_t[None, :] - _tx[:, None], _f0)

# --- amplitude spectrum of the gather (mean over traces), normalized ---
_freq = np.fft.rfftfreq(_nt, _dt)
_spec = np.abs(np.fft.rfft(_gather, axis=1)).mean(axis=0)
_spec /= _spec.max()

fig, ax = plt.subplots(figsize=(14, 4.4))
ax.set_xlim(0, 14); ax.set_ylim(0, 4.4); ax.axis('off'); ax.grid(False)
ax.imshow(np.linspace(0, 1, 256).reshape(-1, 1), extent=[0, 14, 0, 4.4], aspect='auto', origin='lower', zorder=0,
          cmap=LinearSegmentedColormap.from_list('space', ['#04070f', '#0b1622', '#122438', '#0b1622']))
_rng = np.random.default_rng(7)
ax.scatter(_rng.uniform(0, 14, 220), _rng.uniform(0, 4.4, 220), s=_rng.uniform(0.3, 5, 220),
           c='white', alpha=0.45, zorder=1, edgecolors='none')

# --- left: wiggle-trace gather with variable-area fill (time increases downward) ---
_x0, _x1, _ytop, _ybot = 0.35, 3.15, 4.05, 0.35
_xs = np.linspace(_x0 + 0.12, _x1 - 0.12, _ntr)
_dx = (_xs[1] - _xs[0]) * 1.4
_y = _ytop - (_t / _t[-1]) * (_ytop - _ybot)
for _xc, _tr in zip(_xs, _gather):
    _w = _xc + _dx * _tr
    ax.plot(_w, _y, color='#cfe6ff', lw=0.8, zorder=4)
    ax.fill_betweenx(_y, _xc, _w, where=_tr > 0, color=_C['earth'], alpha=0.9, lw=0, zorder=3)

# --- center: continuous chirp with its samples (continuous vs sampled) ---
_xc0, _xc1, _yc = 4.0, 10.0, 2.15
_s = np.linspace(0, 1, 800)
_chirp = lambda s: 0.32 * np.sin(2 * np.pi * (2.0 * s + 5.0 * s ** 2)) * np.sin(np.pi * s) ** 0.5   # 2 -> 12 cycles, tapered ends
ax.plot(_xc0 + (_xc1 - _xc0) * _s, _yc + _chirp(_s), color=_C['continuous'], lw=1.2, alpha=0.9, zorder=4)
_sk = np.linspace(0, 1, 61)
_xk = _xc0 + (_xc1 - _xc0) * _sk
ax.vlines(_xk, _yc, _yc + _chirp(_sk), color=_C['highlight'], lw=1.0, alpha=0.9, zorder=5)
ax.scatter(_xk, _yc + _chirp(_sk), s=9, color=_C['highlight'], zorder=6, edgecolors='none')

# --- right: amplitude spectrum of the gather on the left ---
_sx0, _sx1, _sy0, _sy1 = 10.65, 13.55, 0.45, 3.85
_fmax = 100.0
_m = _freq <= _fmax
_fx = _sx0 + (_sx1 - _sx0) * _freq[_m] / _fmax
_fy = _sy0 + (_sy1 - _sy0) * _spec[_m]
ax.fill_between(_fx, _sy0, _fy, color=_C['freq_domain'], alpha=0.55, lw=0, zorder=3)
ax.plot(_fx, _fy, color='#ffb3a8', lw=1.6, zorder=4)
ax.plot([_sx0, _sx1], [_sy0, _sy0], color='#cfe6ff', lw=1.0, alpha=0.7, zorder=4)
ax.text(_sx1, _sy0 - 0.17, 'f', color='#cfe6ff', fontsize=12, ha='right', va='center', style='italic', zorder=5)
ax.text(_x0 + 0.02, _ybot - 0.17, 't ↓', color='#cfe6ff', fontsize=12, ha='left', va='center', style='italic', zorder=5)

glow_text(ax, 7.0, 3.78, "GPGN 404", 30)
glow_text(ax, 7.0, 3.08, "Digital Signal Processing", 27)
glow_text(ax, 7.0, 1.22, "Professor Jeffrey Shragge", 22)
glow_text(ax, 7.0, 0.58, "Geophysics Dept, Colorado School of Mines", 20)

fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.show()

<center><h1>Module 14 &mdash; Z-Transform</h1></center>

The Fourier transforms of Modules 5 and 11 describe a signal by how much of each complex exponential $e^{i\omega n}$ it contains. The Z-transform
replaces $e^{i\omega}$ by any complex number $z = re^{i\omega}$, so a sequence becomes a polynomial (or power series) in $z^{-1}$. Two things follow: a
delay becomes multiplication by $z^{-1}$, so the difference equations of Module 9 become algebra; and every digital filter is described by the roots of
two polynomials, its zeros and poles. Where those roots sit relative to the unit circle $|z| = 1$ decides how the filter shapes a spectrum and whether it
is stable. This module develops the transform, its region of convergence, and the pole&ndash;zero picture used to design the low-pass, high-pass,
band-pass and band-reject filters of Module 15.

# Outline
- [14.1 The Z-Transform](#14.1-The-Z-Transform)
  - [14.1.1 Connection with the DTFT and the DFT](#14.1.1-Connection-with-the-DTFT-and-the-DFT)
- [14.2 Example 1: Unit Delay](#14.2-Example-1:-Unit-Delay)
- [14.3 Linear Constant-Coefficient Difference Equations](#14.3-Linear-Constant-Coefficient-Difference-Equations)
  - [14.3.1 FIR and IIR systems](#14.3.1-FIR-and-IIR-systems)
  - [14.3.2 Difference equations and differential equations](#14.3.2-Difference-equations-and-differential-equations)
- [14.4 The Fundamental Theorem of Algebra](#14.4-The-Fundamental-Theorem-of-Algebra)
- [14.5 Poles and Zeros](#14.5-Poles-and-Zeros)
  - [14.5.1 Reading the frequency response from the z-plane](#14.5.1-Reading-the-frequency-response-from-the-z-plane)
- [14.6 Thinking About Convergence](#14.6-Thinking-About-Convergence)
- [14.7 Region of Convergence](#14.7-Region-of-Convergence)
  - [14.7.1 Convergence of the causal part](#14.7.1-Convergence-of-the-causal-part)
  - [14.7.2 Convergence of the anticausal part](#14.7.2-Convergence-of-the-anticausal-part)
  - [14.7.3 Combining the results](#14.7.3-Combining-the-results)
  - [14.7.4 Causality and stability](#14.7.4-Causality-and-stability)
- [14.8 Worked Examples 2–6](#14.8-Worked-Examples-2–6)
- [14.9 Finite- and Infinite-Duration Sequences](#14.9-Finite--and-Infinite-Duration-Sequences)
- [14.10 Parting Thoughts](#14.10-Parting-Thoughts)
- [Microquestions](#Microquestions)

<a name="14.1"></a>
## 14.1 The Z-Transform

The [Z-Transform](https://en.wikipedia.org/wiki/Z-transform) is used to convert a **discrete-time** signal (i.e., a sequence of real or complex numbers) into a complex frequency-domain representation.  The Z-Transform, here written symbolically as $\mathcal{Z}[\cdot]$, is commonly given as the **bilateral** or **two-sided** transformation of a discrete time-series $x[n]$ into the [formal power series](https://en.wikipedia.org/wiki/Formal_power_series) $X(z)$ defined as:

<div class="alert alert-info" role="alert">
$$
X(z) = \mathcal{Z} \left\{ x[n] \right\} = \sum_{n=-\infty}^{\infty} x[n] z^{-n} \tag{1}$$ 
</div>

where $n$ is an integer and $z$ is in general a complex number $z=r\mathrm{e}^{i\phi} = r \left( \cos\phi + i\sin\phi \right)$.  For clarity let's explicitly put this expression into the function:

$$
X(z) = \mathcal{Z} \left\{ x[n] \right\} = \sum_{n=-\infty}^{\infty} x[n] r^{-n}e^{-i\phi n} \tag{2}$$


In some cases - and particularly for causal signals (i.e., $x[n]=0$ for $n\lt 0$), one often finds the **unilateral or one-sided Z-Transform**:

<div class="alert alert-info" role="alert">
$$
X(z) = \mathcal{Z} \left\{ x[n] \right\} = \sum_{n=0}^{\infty} x[n] z^{-n} = \sum_{n=0}^{\infty} x[n] r^{-n}e^{-i\phi n} \tag{3}$$
</div>

which is commonly used for evaluating the unit impulse response of a discrete-time **causal** system.

<div class="alert alert-info">
<b>Synthesis &mdash; complex exponentials as eigenfunctions.</b> On the unit circle $z = e^{i\omega}$, and the Z-transform becomes the DTFT of
Section 5.10; sampling that circle at $N$ points gives the DFT of Module 11 (Figure 14.1). The sequence $z^n$ is the discrete eigenfunction of every LTI
system: feed it in and the system multiplies it by $H(z)$. Off the circle, $r = |z|$ adds growth or decay, just as $e^{st}$ with complex $s$ does for
the MATH 225 ODEs.
</div>

<a name="14.1.1"></a>
### 14.1.1 Connection with the DTFT and the DFT

An interesting immediate question is "What is the connection between the
Z-transform and the DFT we studied previously?" The answer is that the Z-transform **reduces** to the DFT for scenarios obeying the following three
conditions:

1. A signal where $x[n]=0$ for $n<0$ and $n \ge N$ (an $N$-point sequence);
2. A $\phi$ defined as $\phi=2\pi k/N$; and
3. A $r$ value defined as $r=1$.

Condition 3 alone puts $z$ on the unit circle, where the Z-transform is the **DTFT**; condition 2 then samples the circle at $N$ equally spaced points,
which gives exactly the **Discrete Fourier Transform (DFT)**. Thus, the **Z-transform** can be seen as a generalization of both. Figure 14.1 shows this for
the sequence $x[n] = [2, 4, 6, 8, 10]$ used in Example 1 below. It turns out that this generalization is very powerful and leads to a whole range of
tools that are useful for filtering!

In [ ]:
C = DSP_COLORS
def zplane(ax, zeros=(), poles=(), roc=None, lim=1.6, unit_circle=True, n=401):
    """Shared z-plane helper: unit circle (dashed), zeros (o), poles (x), and an optional ROC annulus
    roc = (r_in, r_out), shaded; r_out = np.inf for an exterior region."""
    if roc is not None:
        xx = np.linspace(-lim, lim, n); X, Y = np.meshgrid(xx, xx); R = np.hypot(X, Y)
        mask = (R > roc[0]) & (R < roc[1])
        ax.contourf(X, Y, mask.astype(float), levels=[0.5, 1.5], colors=[C['passband']])
        t = np.linspace(0, 2*np.pi, 300)
        for rr in roc:
            if 0 < rr < np.inf:
                ax.plot(rr*np.cos(t), rr*np.sin(t), color=C['time_domain'], lw=1.6)
    if unit_circle:
        t = np.linspace(0, 2*np.pi, 300)
        ax.plot(np.cos(t), np.sin(t), '--', color=C['unit_circle'], lw=1.2)
    ax.axhline(0, color='k', lw=0.6); ax.axvline(0, color='k', lw=0.6)
    zs = np.atleast_1d(np.asarray(zeros, complex)); ps = np.atleast_1d(np.asarray(poles, complex))
    if zs.size:
        ax.plot(zs.real, zs.imag, 'o', mfc='none', mec=C['zero'], ms=10, mew=2, zorder=5)
    if ps.size:
        ax.plot(ps.real, ps.imag, 'x', color=C['pole'], ms=10, mew=2.2, zorder=5)
    ax.set_aspect('equal'); ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel('Re $z$'); ax.set_ylabel('Im $z$')

x = np.array([2, 4, 6, 8, 10.]); N = 5
zk = np.exp(2j*np.pi*np.arange(N)/N)
xx = np.linspace(-2, 2, 401); Xg, Yg = np.meshgrid(xx, xx); Z = Xg + 1j*Yg
with np.errstate(divide='ignore', invalid='ignore'):
    Xz = sum(x[k]*Z**(-k) for k in range(N))
fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 5.2), gridspec_kw=dict(width_ratios=[1, 1.15]))
im = axa.pcolormesh(Xg, Yg, np.log10(np.abs(Xz) + 1e-12), cmap='magma', vmin=-1, vmax=3, shading='auto')
fig.colorbar(im, ax=axa, shrink=0.8, label=r'$\log_{10}|X(z)|$')
zplane(axa, zeros=np.roots(x), poles=[0], lim=2)
axa.plot(zk.real, zk.imag, 's', color=C['highlight'], ms=7, zorder=6, label='DFT points $z_k=e^{i2\\pi k/5}$')
axa.text(0.1, -0.42, '4th-order pole', color='k', fontsize=9, bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.85))
axa.legend(loc='lower left', fontsize=9); axa.set_title('(a) $|X(z)|$ over the $z$-plane')
w = np.linspace(0, 2*np.pi, 1001)
Xw = np.array([np.sum(x*np.exp(-1j*ww*np.arange(N))) for ww in w])
axb.plot(w/np.pi, np.abs(Xw), color=C['freq_domain'], label=r'$|X(e^{i\omega})|$ (DTFT)')
mk, st, bl = axb.stem(2*np.arange(N)/N, np.abs(np.fft.fft(x)), basefmt=' ', label='|DFT|, $N=5$')
plt.setp(st, color=C['highlight']); plt.setp(mk, color=C['highlight'], marker='s')
axb.set_xlim(0, 2); axb.set_ylim(0, 33); axb.set_xlabel(r'$\omega/\pi$ (position around the unit circle)')
axb.set_ylabel('magnitude'); axb.legend(loc='upper center', fontsize=10); axb.set_title('(b) Along the unit circle')
plt.tight_layout(); plt.show()

<b>Figure 14.1.</b> The Z-transform of $x[n]=[2,4,6,8,10]$, $X(z)=2+4z^{-1}+6z^{-2}+8z^{-3}+10z^{-4}$. (a) $\log_{10}|X(z)|$ over the $z$-plane with the unit circle (dashed), the four zeros (blue circles, all outside the unit circle at $|z|$ = 1.445 and 1.547), the fourth-order pole at the origin (red cross) and the five DFT points $z_k=e^{i2\pi k/5}$ (orange squares). (b) $|X|$ along the unit circle (the DTFT, red) and the five-point DFT magnitudes (orange stems), which are samples of it. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; sampling and periodization.</b> Going once around the unit circle returns $z$ to the same point, so $H(e^{i\omega})$ repeats every
$2\pi$. This is the periodic spectrum of a sampled signal from Module 10; $\omega = \pi$ ($z = -1$) is the Nyquist frequency, where the Haar low-pass
filter of Module 13 ($h = [1, 1]/\sqrt{2}$, so $H(z) \propto 1 + z^{-1}$) has its zero.
</div>

<div class="alert alert-info">
<b>Geophysical convention:</b> In geophysics the Z-transform is often written as a power series in $Z = e^{i\omega\Delta t}$, i.e., in positive powers, so that a
causal sequence is a polynomial in $Z$ (for example, in Claerbout's books). The two conventions are equivalent, but every statement about "inside" and
"outside" the unit circle flips between them. Because most of the DSP literature uses powers of $z^{-1}$, these notes do too.
</div>

<a name="14.2"></a>
## 14.2 Example 1: Unit Delay

**Q:** What happens when we take the Z-transform of signal $x[n]$ ($n\ge0$) that has been delayed by $k$ samples (i.e., $x[n-k]$)?

**A:** Let's evaluate this using the definition of the unilateral or one-sided Z-transform.

$$
\begin{array}{ll}
\mathcal{Z}\left\{x[n-k]\right\}
&=&
\sum_{n=0}^{\infty} x[n-k]z^{-n}  \\
\,&=&
\sum_{j=-k}^{\infty} x[j]z^{-(j+k)}\quad \quad j=n-k  \\
\,&=&
\sum_{j=-k}^{\infty} x[j]z^{-j}z^{-k}  \\
\,&=&
z^{-k}\sum_{j=-k}^{\infty} x[j]z^{-j} \\
\,&=&
z^{-k}\sum_{j=0}^{\infty} x[j]z^{-j}\quad\quad x[j]=0, j<0  \\
\,&=&
z^{-k}X(z)  \\
\end{array}\tag{4}
$$

where $X(z)$ is the Z-transform of $x[n]$.  Thus, $z^{-k}$ can be interpreted as an operator that **delays** a sequence by $k$ samples. Correspondingly, $z^k$ is an operator that **advances** a sequence by $k$ samples.

A consequence of this interpretation is that it can be used to represent regularly sampled time-series. For example, if we have a sequence of numbers $x[n]=[2,4,6,8,10]$ for $n=[0,1,2,3,4]$ then we can think about representing this sequence as the following $X(z)$ time-series:

$$X(z) = 2z^{0} + 4z^{-1} + 6z^{-2} + 8z^{-3} + 10z^{-4} \tag{5} $$ 

where the nth term in the advancing time series corresponds to the power $z^{-n}$.

<a name="14.3"></a>
## 14.3 Linear Constant-Coefficient Difference Equations

In Section 9.8 we met linear constant-coefficient difference equations (LCCDE) in the context of linear time-invariant (LTI) systems.  In any LTI system, its input $x[n]$ and output $y[n]$ can be related via a Nth order linear constant coefficient [difference equation](https://en.wikipedia.org/wiki/Linear_difference_equation):

$$ \sum_{k=0}^{N} a_k y[n-k] = \sum_{k=0}^{M} b_k x[n-k] \tag{6} $$

Let's explore what happens when we apply the Z-transform concept that we discussed above.  Applying it to both sides of equation (6) yields the following

$$\mathcal{Z} \left[ \sum_{k=0}^{N} a_k y[n-k] \right] 
=  
\mathcal{Z} \left[ \sum_{k=0}^{M} b_k x[n-k] \right] \tag{7}$$

However, because this is a linear operation we may write

$$ \sum_{k=0}^{N} a_k \mathcal{Z} \left[ y[n-k] \right] 
=  
\sum_{k=0}^{M} b_k \mathcal{Z} \left[  x[n-k] \right] \tag{8}$$

and then apply the definition of the Z-transform as the unit delay operator

$$\sum_{k=0}^N a_k z^{-k}Y(z) =  \sum_{k=0}^{M} b_k z^{-k} X(z) \tag{9}$$

Realizing that $Y(z)$ and $X(z)$ do not depend on $k$ allows us to write:

$$Y(z) \sum_{k=0}^N a_k z^{-k} = X(z) \sum_{k=0}^{M} b_k z^{-k} \tag{10}$$

Let's now rewrite this equation by dividing both sides of equation (10) by $X(z)$ and by the series on the left.  This results in the following expression:

$$H(z) \equiv \frac{Y(z)}{X(z)} =  \frac{\sum_{k=0}^{M} b_k z^{-k}}{\sum_{k=0}^N a_k z^{-k} } \tag{11} $$

where $H(z)$ is known as the [transfer function](https://en.wikipedia.org/wiki/Transfer_function) and effectively describes how the input and output are related in Z-space:

$$Y(z) = H(z) X(z) \tag{12}$$

Applying $H(z)$ may also be thought of as applying a **filtering** operation that transforms an input signal represented by $X(z)$ into an output signal represented by $Y(z)$.  

We have seen something like this before when we discussed the convolution theorem.  In the case where the Z-transform becomes a Fourier Transform the operation in equation (12) effectively represents the **convolution theorem in the frequency domain**.  Thus, this may be thought of as an extension of the convolution theorem to the Z-transform.

<div class="alert alert-info">
<b>Synthesis &mdash; convolution and multiplication.</b> Multiplying two polynomials in $z^{-1}$ collects the same products as the discrete convolution
sum of Section 9.7: $(1 + 2z^{-1})(3 - z^{-1}) = 3 + 5z^{-1} - 2z^{-2}$ has the coefficients of <code>np.convolve([1, 2], [3, -1])</code> $= [3, 5, -2]$.
$Y(z) = H(z)X(z)$ is the convolution theorem of Module 7 written for sequences (MATH 213: polynomial products).
</div>

<a name="14.3.1"></a>
### 14.3.1 FIR and IIR systems

There are two classes of transfer function that are commonly applied in digital signal processing:

1. A system with coefficients $a_k=0, k\ge 1$ does not have feedback, and is called a **nonrecursive filter** or a [finite-impulse response (FIR)](https://en.wikipedia.org/wiki/Finite_impulse_response) filter.

2. A system with coefficients $a_k\neq 0, k\ge 1$ is said to have **feedback** since the current output value depends on the *previous* output values.  A filter exhibiting such a characteristic is called a **recursive filter** or an [infinite-impulse response (IIR)](https://en.wikipedia.org/wiki/Infinite_impulse_response) filter. 

Let's now make a connection with the LCCDE studied previously: 

1. An averaging filter depends only on the input sequence and not on the previous filtered output and thus an averaging operator may be considered as a FIR filtering operator.

2. Your current bank account or mortgage balance depends on what it was in previous months, and thus compound interest or mortgage amortization operators may be considered as a recursive IIR filtering operator.

<a name="14.3.2"></a>
### 14.3.2 Difference equations and differential equations

The damped oscillator of MATH 225 and PHGN 100 (a mass on a spring, or
a seismometer),

$$\ddot{y} + 2\zeta\omega_n\dot{y} + \omega_n^2 y = x(t) \tag{13}$$

has free solutions $e^{st}$ whose exponents are the roots of the characteristic equation $s^2 + 2\zeta\omega_n s + \omega_n^2 = 0$, namely
$s = -\zeta\omega_n \pm i\omega_d$ with $\omega_d = \omega_n\sqrt{1-\zeta^2}$. Sample the solution every $T$ seconds: $e^{st}$ at $t = nT$ is
$(e^{sT})^n = z^n$ with

$$z = e^{sT} = e^{-\zeta\omega_n T}\, e^{\pm i\omega_d T} \tag{14}$$

so the sampled oscillation satisfies a second-order LCCDE $y[n] + a_1 y[n-1] + a_2 y[n-2] = 0$ whose characteristic roots are exactly these $z$
($a_1 = -2r\cos\theta$, $a_2 = r^2$, with $r = e^{-\zeta\omega_n T}$ and $\theta = \omega_d T$).

<div class="alert alert-info">
<b>Note:</b> <b>Difference equations are sampled differential equations.</b> The exponential trial solution of MATH 225 becomes the power $z^n$, and
the left half of the $s$-plane (Re $s<0$, decaying solutions) maps inside the unit circle ($|z| = e^{\mathrm{Re}\,s\,T} < 1$). Figure 14.2 checks this
numerically: the recursion and the sampled ODE solution agree to rounding error.
</div>

In [ ]:
C = DSP_COLORS
fn, zeta, T = 2.0, 0.1, 0.02                     # natural frequency (Hz), damping ratio, sample interval (s)
wn = 2*np.pi*fn; wd = wn*np.sqrt(1 - zeta**2)
s = np.array([-zeta*wn + 1j*wd, -zeta*wn - 1j*wd])
z = np.exp(s*T)
a1, a2 = -2*np.abs(z[0])*np.cos(np.angle(z[0])), np.abs(z[0])**2
n = np.arange(60); t = np.linspace(0, n[-1]*T, 600)
y_ode = np.exp(-zeta*wn*t)*np.cos(wd*t)
y = np.zeros(n.size); y[0] = 1.0; y[1] = np.exp(-zeta*wn*T)*np.cos(wd*T)
for k in range(2, n.size):
    y[k] = -a1*y[k-1] - a2*y[k-2]
fig = plt.figure(figsize=(14, 4.4))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 2.0])
ax1, ax2, ax3 = fig.add_subplot(gs[0]), fig.add_subplot(gs[1]), fig.add_subplot(gs[2])
ax1.axhline(0, color='k', lw=0.6); ax1.axvline(0, color='k', lw=0.6)
ax1.plot(s.real, s.imag, 'x', color=C['pole'], ms=11, mew=2.2)
ax1.axvspan(-20, 0, color=C['passband'], alpha=0.6)
ax1.set_xlim(-16, 6); ax1.set_ylim(-16, 16); ax1.set_aspect('equal')
ax1.set_xlabel(r'Re $s$ [s$^{-1}$]'); ax1.set_ylabel(r'Im $s$ [rad/s]'); ax1.set_title('(a) $s$-plane: Re $s<0$ decays')
zplane(ax2, poles=z, roc=(0, 1), lim=1.25)
ax2.set_title(r'(b) $z=e^{sT}$: inside $|z|<1$')
ax3.plot(t, y_ode, color=C['continuous'], lw=1.5, label='sampled ODE solution $e^{-\\zeta\\omega_n t}\\cos\\omega_d t$')
mk, st, bl = ax3.stem(n*T, y, basefmt=' ', label=f'LCCDE: $y[n]={-a1:.4f}\\,y[n-1]-{a2:.4f}\\,y[n-2]$')
plt.setp(st, color=C['output']); plt.setp(mk, color=C['output'], ms=4)
ax3.set_xlabel('t [s]'); ax3.set_ylabel('y'); ax3.set_xlim(0, n[-1]*T); ax3.set_ylim(-1.05, 1.3)
ax3.legend(loc='upper right', fontsize=9); ax3.set_title(f'(c) $f_n$ = {fn:.0f} Hz, $\\zeta$ = {zeta}, $T$ = {T} s')
plt.tight_layout(); plt.show()
print(f'max |recursion - sampled ODE| = {np.max(np.abs(y - np.exp(-zeta*wn*n*T)*np.cos(wd*n*T))):.1e}')

<b>Figure 14.2.</b> From a differential equation to a difference equation, for $f_n$ = 2 Hz, $\zeta$ = 0.1 and $T$ = 0.02 s. (a) The two characteristic roots $s$ in the left half of the $s$-plane (shaded). (b) The discrete poles $z=e^{sT}$, at radius $r = 0.9752$ inside the unit circle (shaded disk). (c) The sampled ODE solution (gray) and the LCCDE recursion with $a_1 = -1.8897$, $a_2 = 0.9510$ (stems), started from the first two samples; the two agree to better than $10^{-12}$. <i>(Python-generated.)</i>

<a name="14.4"></a>
## 14.4 The Fundamental Theorem of Algebra

The [Fundamental Theorem of Algebra](https://en.wikipedia.org/wiki/Fundamental_theorem_of_algebra) states that 

"Every non-zero, single-variable, degree n polynomial with complex coefficients has, counted with multiplicity, exactly n complex roots. The equivalence of the two statements can be proven through the use of successive polynomial division."

What is the implication here? It means that an $n$th-order polynomial,

$$p(z)=\sum_{k=0}^{n}c_k z^k = c_0+c_{1}z+c_{2}z^{2}+\cdots +c_{n}z^{n} \tag{15}$$

can be rewritten as

$$p(z)= c_n \prod_{k=1}^n (z-z_k) = c_n(z-z_{1})(z-z_{2})\cdots (z-z_{n}) \tag{16}$$

where $\prod$ is product notation [e.g., $\prod_{k=3}^{5} k =3 \times 4 \times 5=60$]. Applying this to the numerator and denominator of equation (11) (after multiplying both by $z^{\max(M,N)}$ to clear the negative powers), we can write the transfer function $H(z)$ as:

<div class="alert alert-info">
$$H(z) \equiv \frac{Y(z)}{X(z)} = \frac{b_0}{a_0}\, z^{N-M}\, \frac{\prod_{k=1}^{M} (z-z_k)}{\prod_{k=1}^{N}(z-p_k) } \tag{17} $$
</div>

You may (should!) be concerned about what's going on in the denominator.  In particular, what is happening whenever $z=p_k$: division by zero!  Thus, we must examine the **stability** of these transfer functions as well as determine in which region of the complex plane these series converge.

<a name="14.5"></a>
## 14.5 Poles and Zeros

Starting from the $H(z)$ defined in equation (17), let's examine a few different important values.

**Zeros:** are the values of $z$ for which $H(z)=0$.  These occur at $z=z_k$ in equation (17).  These **do not cause** any stability issues; however, they cause certain parts of the output spectrum to be equal to zero.

**Poles:** are the values of $z$ for which $H(z)=\infty$.  These occur at $z=p_k$ in equation (17). These **do cause** stability issues since division by zero will cause an infinite output!

An important concept is that by choosing where we put the zeros $z_k$ and poles $p_k$ we can design filters (e.g., low-pass, high-pass, band-pass) that will take our input data $x[n]$ and give us the desired output signals $y[n]$ (e.g., low-passed, high-passed or band-passed data).

**Q:** How many zeros and poles are there in equation (17)?

**A:** There are $M$ zeros and $N$ poles in the finite plane, plus $|N-M|$ more at $z=0$ from the factor $z^{N-M}$ (zeros if $N>M$, poles if $M>N$).

<a name="14.5.1"></a>
### 14.5.1 Reading the frequency response from the z-plane

On the unit circle $z = e^{i\omega}$, and each factor of
equation (17) is a vector from a zero or a pole to the point $e^{i\omega}$. Since $|e^{i\omega(N-M)}| = 1$, the magnitude response is a ratio of
vector lengths:

$$|H(e^{i\omega})| = \left|\frac{b_0}{a_0}\right| \frac{\prod_{k=1}^{M} |e^{i\omega}-z_k|}{\prod_{k=1}^{N}|e^{i\omega}-p_k|} \tag{18}$$

As $e^{i\omega}$ passes close to a pole, a short vector in the denominator makes $|H|$ large; passing over a zero on the circle makes $|H|$ exactly zero.

The animation in Figure 14.3 moves the point $e^{i\omega}$ (black dot) from $\omega = 0$ to $\omega = \pi$ around the upper half of the
unit circle for a filter with zeros on the circle at $\omega = \pm 0.7\pi$ and poles at radius 0.9 and angle $\pm 0.3\pi$. Watch the dashed vector to the
upper pole: as it shortens near $\omega = 0.3\pi$, $|H|$ rises to its peak of 15.4. Then watch the solid vector to the upper zero shrink to
nothing at $\omega = 0.7\pi$, where $|H|$ drops to zero. Between these, the response changes smoothly, and at $\omega=\pi$ it is only 0.29.

In [ ]:
C = DSP_COLORS
zeros = np.exp(1j*0.7*np.pi*np.array([1, -1]))          # zeros on the unit circle at +-0.7 pi
poles = 0.9*np.exp(1j*0.3*np.pi*np.array([1, -1]))      # poles at radius 0.9, +-0.3 pi
b, a = np.real(np.poly(zeros)), np.real(np.poly(poles))
w = np.linspace(0, np.pi, 1001)
E = np.exp(-1j*np.outer(w, np.arange(3)))
H = np.abs((E @ b)/(E @ a))                                # |H(e^{iw})| from the coefficients
n_frames = 30
wf = np.linspace(0, np.pi, n_frames)
th = np.linspace(0, 2*np.pi, 300)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 4.6), gridspec_kw=dict(width_ratios=[1, 1.35]))
ax1.plot(np.cos(th), np.sin(th), '--', color=C['unit_circle'], lw=1)
ax1.axhline(0, color='k', lw=0.6); ax1.axvline(0, color='k', lw=0.6)
ax1.plot(zeros.real, zeros.imag, 'o', mfc='none', mec=C['zero'], ms=11, mew=2)
ax1.plot(poles.real, poles.imag, 'x', color=C['pole'], ms=11, mew=2.2)
zl = [ax1.plot([], [], color=C['zero'], lw=1.6)[0] for _ in zeros]
pl = [ax1.plot([], [], color=C['pole'], lw=1.6, ls='--')[0] for _ in poles]
pt, = ax1.plot([], [], 'ko', ms=8, zorder=6)
ax1.set_aspect('equal'); ax1.set_xlim(-1.2, 1.2); ax1.set_ylim(-1.2, 1.2)
ax1.set_xlabel('Re $z$'); ax1.set_ylabel('Im $z$'); ax1.set_title('(a) $z$-plane: vectors from zeros (solid) and poles (dashed)', fontsize=10)
ax2.plot(w/np.pi, H, color=C['continuous'], lw=1, alpha=0.5)
tr, = ax2.plot([], [], color=C['freq_domain'], lw=2)
dot, = ax2.plot([], [], 'ko', ms=7)
ax2.set_xlim(0, 1); ax2.set_ylim(0, 1.08*H.max())
ax2.set_xlabel(r'$\omega/\pi$'); ax2.set_ylabel(r'$|H(e^{i\omega})|$'); ax2.set_title('(b) Magnitude response traced so far', fontsize=10)
title = fig.suptitle('', fontsize=13)
state = {}

def update(k):
    p = np.exp(1j*wf[k])
    for line, q in zip(zl, zeros):
        line.set_data([q.real, p.real], [q.imag, p.imag])
    for line, q in zip(pl, poles):
        line.set_data([q.real, p.real], [q.imag, p.imag])
    pt.set_data([p.real], [p.imag])
    Hk = np.prod(np.abs(p - zeros))/np.prod(np.abs(p - poles))   # equation: product of vector lengths
    m = w <= wf[k] + 1e-12
    tr.set_data(w[m]/np.pi, H[m]); dot.set_data([wf[k]/np.pi], [Hk])
    title.set_text(f'$\\omega$ = {wf[k]/np.pi:.2f}$\\pi$:   $|H|$ = (product of solid lengths)/(product of dashed lengths) = {Hk:.2f}')
    state.update(k=k, Hk=Hk)
    return []

update(0)
plt.tight_layout()
ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=300, blit=False, repeat=True)
plt.close(fig)
HTML('<div style="display:flex; justify-content:center;">' + ani.to_jshtml() + '</div>')

<b>Figure 14.3.</b> Reading $|H(e^{i\omega})|$ from the $z$-plane (equation 18), 30 frames for $\omega$ from 0 to $\pi$. (a) Zeros (blue circles) at $e^{\pm i0.7\pi}$, poles (red crosses) at $0.9e^{\pm i0.3\pi}$, and the vectors from each to $e^{i\omega}$; $b = [1, 1.1756, 1]$, $a = [1, -1.0580, 0.81]$. (b) $|H|$ traced up to the current $\omega$ (red) over the full response (gray): $|H(0)|$ = 4.22, a peak of 15.38 at $\omega = 0.296\pi$, zero at $0.7\pi$ and 0.29 at $\pi$. <i>(Python-generated.)</i>

<a name="14.6"></a>
## 14.6 Thinking About Convergence

Let's first refresh ourselves about the concept of convergence and (infinite) geometric series.  We can start with the following $N$ term **finite geometric series**:

$$y = \sum_{n=0}^{N-1} ax^n = a \left(\frac{1-x^N}{1-x}\right) \tag{19}$$

where we have used a [geometric sum](https://en.wikipedia.org/wiki/Geometric_progression#Geometric_series) to evaluate on the right hand side. There are no issues here in terms of stability (i.e., $y<\infty$) even at $x=1$ because this just results in a sum of $y=aN$.

What happens when we now consider an **infinite geometric series**?

$$y = \sum_{n=0}^{\infty} ax^n = \lim_{N\rightarrow\infty} a \left(\frac{1-x^N}{1-x}\right) \tag{20}$$

Clearly, there is now a different concern: for $y<\infty$ we need $|x|<1$, because then $x^N \rightarrow 0$ and $y = a/(1-x)$; for $|x| \ge 1$ the terms do not shrink and the sum does not converge. The restriction $-1< x < 1$ is thus the **region of convergence** of this infinite sum along the real $x$ axis.

What happens if we now have the following **infinite geometric series**?

$$y = \sum_{n=0}^{\infty} (ax)^n = \frac{1}{1-ax} \quad \mathrm{for}\ |ax|<1 \tag{21}$$

In this case we need $|ax|<1$, or better yet $|a||x|<1$.  Thus, for this to be stable (i.e., $y<\infty$) we need to have $|x|< |a|^{-1}$.

<a name="14.7"></a>
## 14.7 Region of Convergence

The region of convergence (ROC) indicates where the Z-transform of a sequence converges.  Generally, there exists some $z$ such that 

$$\left| X(z) \right| =\left|\sum_{n=-\infty}^{\infty} x[n]z^{-n}\right|\rightarrow \infty \tag{22}$$

where the Z-transform **does not converge**.  The set of values for $z$ for which $X(z)$ converges,

<div class="alert alert-info">
$$\left| X(z) \right| =\left|\sum_{n=-\infty}^{\infty} x[n]z^{-n}\right| \le \sum_{n=-\infty}^{\infty} \left|x[n]z^{-n}\right|   < \infty \tag{23}$$
</div>

is called the ROC.  The ROC must be specified along with $X(z)$ in order for the Z-transform to be considered "complete".

Assuming that $x[n]$ is of infinite length, let's decompose $X(z)$ as the following:

$$\begin{eqnarray}
X(z) &=& X_{-}(z) + X_+(z) \tag{24}
\end{eqnarray}$$

where these two contributions are the anticausal ($X_-(z)$) and causal ($X_+(z)$) components of $X(z)$:

$$\begin{eqnarray}
X_{-}(z) &=& \sum_{n=-\infty}^{-1}x[n]z^{-n} \tag{25}\\
X_{+}(z) &=& \sum_{n=0}^{\infty}x[n]z^{-n} \tag{26}\\
\end{eqnarray}$$

where it is clear that the sum of equations (25) and (26) satisfies equation (24).

<a name="14.7.1"></a>
### 14.7.1 Convergence of the causal part

The causal part $X_+(z)$:

For a series to converge, the series

$$\begin{eqnarray}
X_+(z) &=& x[0]z^{-0} + x[1]z^{-1} +\dots + x[n]z^{-n}+ ... \tag{27}\\
\,     &=& f_0(z)+f_1(z)+ \dots + f_n(z) + \dots \tag{28}\\
\end{eqnarray}$$

has to satisfy the following [ratio test](https://en.wikipedia.org/wiki/Ratio_test) (MATH 213):

$$\lim_{n\rightarrow\infty} 
\left|
\frac{f_{n+1}(z)}{f_n(z)}
\right|
< 1 \tag{29}
$$

Thus, assuming that the ratio of successive terms of $x[n]$ converges to a finite value $R_+$

$$
\lim_{n\rightarrow\infty} 
\left|
\frac{x[n+1]}{x[n]}
\right|
= R_+ \tag{30}
$$

then $X_+(z)$ will converge if

$$
\lim_{n\rightarrow\infty} 
\left|
\frac{x[n+1]z^{-n-1}}{x[n]z^{-n}}
\right|
=
\lim_{n\rightarrow\infty} 
\left|
\frac{x[n+1]}{x[n]}
\right|
\left|z^{-1}\right|
<1 \tag{31}
$$
This implies that

$$\left|z\right| > \lim_{n\rightarrow\infty} 
\left|
\frac{x[n+1]}{x[n]}
\right|=R_+ \tag{32}
$$

That is, the ROC for $X_+(z)$ is 

$$|z|>R_+ \tag{33}$$

<a name="14.7.2"></a>
### 14.7.2 Convergence of the anticausal part

The anticausal part $X_-(z)$:

Assuming that the ratio of successive terms (moving to the left) converges to a finite value $1/R_-$

$$
\lim_{m\rightarrow\infty} 
\left|
\frac{x[-m-1]}{x[-m]}
\right|
= \frac{1}{R_-} \tag{34}
$$

the anticausal components will converge if

$$
\lim_{m\rightarrow\infty} 
\left|
\frac{x[-m-1]z^{m+1}}{x[-m]z^{m}}
\right|
=
\lim_{m\rightarrow\infty} 
\left|
\frac{x[-m-1]}{x[-m]}
\right|
\left|z\right| = \frac{1}{R_-} \left|z\right|
<1 \tag{35}
$$

Thus, the ROC for $X_-(z)$ is 

$$|z|<R_- \tag{36}$$

<a name="14.7.3"></a>
### 14.7.3 Combining the results

The ROC of $X(z)$ for an infinite two-sided sequence is given by 

$$R_+<|z|<R_- \tag{37}$$

Note that if $R_- < R_+$ then there is **no ROC** and $X(z)$ **does not exist**.

Let's look at this graphically in Figure 14.4.

In [ ]:
C = DSP_COLORS
Rp, Rm = 0.6, 1.4
cases = [((Rp, np.inf), '(a) $X_+(z)$: $|z|>R_+$'), ((0, Rm), '(b) $X_-(z)$: $|z|<R_-$'),
         ((Rp, Rm), '(c) $X(z)$: $R_+<|z|<R_-$')]
fig, axs = plt.subplots(1, 3, figsize=(13.5, 4.6))
for ax, (roc, ttl) in zip(axs, cases):
    zplane(ax, roc=roc, lim=1.8)
    ax.set_title(ttl, fontsize=12)
    if roc[0] > 0:
        ax.annotate('$R_+$', xy=(Rp*np.cos(2.4), Rp*np.sin(2.4)), xytext=(-1.6, 1.4), fontsize=12,
                    arrowprops=dict(arrowstyle='->', color='k', lw=0.8))
    if roc[1] < np.inf:
        ax.annotate('$R_-$', xy=(Rm*np.cos(-0.7), Rm*np.sin(-0.7)), xytext=(1.05, -1.65), fontsize=12,
                    arrowprops=dict(arrowstyle='->', color='k', lw=0.8))
plt.tight_layout(); plt.show()

<b>Figure 14.4.</b> Regions of convergence (shaded), drawn for $R_+ = 0.6$ and $R_- = 1.4$ with the unit circle dashed. (a) The causal part $X_+(z)$ converges outside a circle, $|z|>R_+$. (b) The anticausal part $X_-(z)$ converges inside a circle, $|z|<R_-$. (c) A two-sided sequence converges in the annulus where both do, $R_+<|z|<R_-$. <i>(Python-generated.)</i>

<a name="14.7.4"></a>
### 14.7.4 Causality and stability

Section 9.6.5 defined a stable (bounded-input, bounded-output) system as one whose impulse
response is absolutely summable, $\sum_n |h[n]| < \infty$. On the unit circle $|z^{-n}| = 1$, so that is exactly the condition for $H(z)$ to converge at
$|z|=1$ (equation 23): **a system is stable if and only if the ROC of $H(z)$ contains the unit circle.** For a causal system the ROC lies outside
its outermost pole (Section 14.7.1), which gives the rule used in Module 15:

$$\text{causal and stable} \iff |p_k| < 1 \ \text{for every pole } p_k \tag{38}$$

<div class="alert alert-info">
<b>Synthesis &mdash; systems as operators.</b> The bank balance of Module 9 (Example 10) obeys $y[n] = (1 + \alpha p/100)\, y[n-1]$, i.e.,
$H(z) = 1/(1 - (1 + \alpha p/100)z^{-1})$: a single pole at $1 + \alpha p/100 > 1$, outside the unit circle. The response grows without bound, which is
good for savers and bad for filters.
</div>

<div class="alert alert-success">
<h3><center>Distraction A: Minimum phase: where a seismic wavelet's zeros live</center></h3>

A seismic wavelet $w[n]$ is itself a polynomial $W(z)$, and its inverse
filter $1/W(z)$ has poles where $W$ has zeros. If every zero of $W(z)$ lies inside the unit circle, the inverse filter is causal and stable, and the
wavelet is called <i>minimum phase</i>. This is the assumption behind predictive deconvolution in seismic processing (Peacock and Treitel, 1969; see the SEG Wiki entry
<a href="https://wiki.seg.org/wiki/Predictive_deconvolution">Predictive deconvolution</a>).
When a zero lies outside the circle, the causal inverse grows without bound, the same instability met in the deconvolution of Section 7.9 and the
ill-conditioning of Module 8. A stable inverse then exists only as an anticausal sequence. Microquestion 14.1 puts numbers on this.

<b>Reference.</b> Peacock, K. L., and S. Treitel (1969), Predictive deconvolution: theory and practice, <i>Geophysics</i> 34(2), 155&ndash;169.
</div>

<a name="14.8"></a>
## 14.8 Worked Examples 2–6

The following examples work out the Z-transform and ROC of the most common sequences.

### Example 2: causal geometric sequence

**Q:** Determine the z-transform of $x[n] = a^n u[n]$ where $u[n]$ is the unit step function. 

**A:** The input function may be written as

$$X(z) = \sum_{n=-\infty}^{\infty} a^n u[n]z^{-n} = \sum_{n=0}^{\infty} a^n z^{-n}= \sum_{n=0}^{\infty} \left(a z^{-1}\right)^n \tag{39}$$

According to the above, $X(z)$ will converge if

$$\sum_{n=0}^{\infty} \left|a z^{-1}\right|^n< \infty \tag{40}$$

Applying the **ratio test**

$$\lim_{n\rightarrow\infty} 
\left|
\frac{f_{n+1}(z)}{f_n(z)}
\right|
< 1 \tag{41}
$$

we get

$$\lim_{n\rightarrow\infty} 
\left|
\frac{a^{n+1}z^{-n-1}}{a^nz^{-n}}
\right|
= 
\left|
az^{-1}
\right|
< 1 \tag{42}
$$

Thus, the convergence condition is that $|a|<|z|$.  Thus, within this region we may use a geometric series to evaluate this infinite summation in the region where the series converges:

$$
X(z) = \sum_{n=0}^\infty \left(az^{-1}\right)^n
=
\frac{1-\left(az^{-1}\right)^\infty}{1-az^{-1}}
=
\frac{1}{1-az^{-1}}
=
\frac{z}{z-a}
\tag{43}
$$
where the second equality is reached in the ROC because $|az^{-1}|<1$, which means that $(az^{-1})^\infty=0$; the final step is just multiplying top and bottom by $z$.

Thus, together with the ROC, the z-transform of $x[n]=a^n u[n]$ is:

$$X(z) = \frac{z}{z-a}, \quad |a| < |z| \tag{44}$$

It is clear that $X(z)$ has a **zero** at $z=0$ and a **pole** at $z=a$.

Figure 14.5 shows the sequence and its ROC for one value of $a$ inside and one outside the unit circle.

In [ ]:
C = DSP_COLORS
n = np.arange(-5, 16)
fig, axs = plt.subplots(2, 2, figsize=(11, 8.6), gridspec_kw=dict(height_ratios=[1, 1.5]))
for j, a in enumerate([0.7, 1.3]):
    x = np.where(n >= 0, a**np.clip(n, 0, None), 0.0)
    mk, st, bl = axs[0, j].stem(n, x, basefmt='k-')
    plt.setp(st, color=C['time_domain']); plt.setp(mk, color=C['time_domain'], ms=5)
    axs[0, j].set_xlabel('$n$'); axs[0, j].set_ylabel('$x[n]$'); axs[0, j].set_xlim(-5.5, 15.5)
    axs[0, j].set_title(f'({"ab"[j]}) $x[n]=({a})^n u[n]$')
    zplane(axs[1, j], zeros=[0], poles=[a], roc=(a, np.inf), lim=1.8)
    inside = a < 1
    axs[1, j].set_title(f'({"cd"[j]}) ROC $|z|>{a}$: unit circle ' + ('inside the ROC' if inside else 'outside the ROC'), fontsize=11)
plt.tight_layout(); plt.show()

<b>Figure 14.5.</b> Example 2, $x[n]=a^n u[n]$. (a, b) The sequence for $a$ = 0.7 (decaying) and $a$ = 1.3 (growing). (c, d) Its ROC $|z|>|a|$ (shaded), with the zero at the origin and the pole at $z=a$. For $|a|<1$ the ROC contains the unit circle (dashed), so the DTFT (the Z-transform on the unit circle) exists; for $|a|>1$ it does not. <i>(Python-generated.)</i>

### Example 3: anticausal geometric sequence

**Q:** Determine the z-transform of $x[n]=-b^n u[-n-1]$.

**A:** Let's write

$$X(z) =
-\sum_{n=-\infty}^{\infty}b^n u[-n-1] z^{-n}
= 
-\sum_{n=-\infty}^{-1}b^n z^{-n}\tag{45}
$$

Let's now identify $m=-n$ and write

$$X(z) =
-\sum_{m=1}^{\infty}b^{-m} z^{m}
=
-\sum_{m=1}^{\infty}\left(b^{-1} z\right)^{m} \tag{46}
$$

Thus, like the Example 2, $X(z)$ converges if $\left|b^{-1}z\right|<1$, or $|b|>|z|$.  This gives:

$$X(z)=-\sum_{m=1}^{\infty} \left(b^{-1}z\right)^m = 
- \frac{b^{-1}z(1-(b^{-1}z)^\infty)}{1-b^{-1}z} =- \frac{b^{-1}z}{1-b^{-1}z} = - \frac{z}{b-z} = \frac{z}{z-b} \tag{47}$$

Thus, together with the ROC, the z-transform of $x[n]=-b^n u[-n-1]$ is:

$$X(z) = \frac{z}{z-b}, \quad |b|>|z| \tag{48}$$

Again, it is clear that $X(z)$ has a **zero** at $z=0$ and a **pole** at $z=b$.

Figure 14.6 shows the same comparison for the anticausal sequence.

In [ ]:
C = DSP_COLORS
n = np.arange(-15, 6)
fig, axs = plt.subplots(2, 2, figsize=(11, 8.6), gridspec_kw=dict(height_ratios=[1, 1.5]))
for j, b in enumerate([0.7, 1.3]):
    x = np.where(n <= -1, -b**np.clip(n, None, -1).astype(float), 0.0)
    mk, st, bl = axs[0, j].stem(n, x, basefmt='k-')
    plt.setp(st, color=C['time_domain']); plt.setp(mk, color=C['time_domain'], ms=5)
    axs[0, j].set_xlabel('$n$'); axs[0, j].set_ylabel('$x[n]$'); axs[0, j].set_xlim(-15.5, 5.5)
    axs[0, j].set_title(f'({"ab"[j]}) $x[n]=-({b})^n u[-n-1]$')
    zplane(axs[1, j], zeros=[0], poles=[b], roc=(0, b), lim=1.8)
    inside = b > 1
    axs[1, j].set_title(f'({"cd"[j]}) ROC $|z|<{b}$: unit circle ' + ('inside the ROC' if inside else 'outside the ROC'), fontsize=11)
plt.tight_layout(); plt.show()

<b>Figure 14.6.</b> Example 3, $x[n]=-b^n u[-n-1]$. (a, b) The sequence for $b$ = 0.7 and $b$ = 1.3 (nonzero only for $n \le -1$). (c, d) Its ROC $|z|<|b|$ (shaded), with the zero at the origin and the pole at $z=b$. Now the unit circle lies in the ROC, and the DTFT exists, only when $|b|>1$. <i>(Python-generated.)</i>

### Example 4: combining Examples 2 and 3

**Q:** Determine the z-transform of $x[n] = a^n u[n]+b^n u[-n-1]$ where $|a|<|b|$.

**A:** Employing the previous results (note the sign: Example 3 has $-b^n u[-n-1]$) we have

$$
\begin{eqnarray}
X(z) &=& \frac{z}{z-a}-\frac{z}{z-b} \quad |a|<|z|<|b| \tag{49} \\
&=&
\frac{(a-b)z}{(z-a)(z-b)}, \quad |a|<|z|<|b| \tag{50}\\
\end{eqnarray}
$$

Note that there is still one **zero** ($z=0$) but there are now two **poles** ($z=a$ and $z=b$).  Note that the ROC is the annulus of Figure 14.4(c), with $|a|=R_+$ and $|b|=R_-$. 

### Example 5: unit impulse

**Q:** What is the Z-transform of $x[n]=\delta[n-p]$.

**A:** We have

$$X(z) = \sum_{n=-\infty}^\infty \delta[n-p]z^{-n} = z^{-p} \tag{51}$$

which converges wherever $z^{-p}$ is finite: for $p>0$ (a delay) the ROC is $|z|>0$, all $z$ except the origin; for $p<0$ (an advance) it is $|z|<\infty$, all finite $z$; and for $p=0$ it is the whole plane.

### Example 6: finite geometric series

**Q:** Determine the z-transform of $x[n]$ which has the form

$$x[n] = \left\{
\begin{array}{cc}
a^n & 0\le n\le N-1 \\
0 & \mathrm {otherwise}\\
\end{array}
\right. \tag{52}
$$

**A:** Assuming that $a^N<\infty$, we have by the geometric series

$$X(z) = \sum_{n=0}^{N-1}\left(az^{-1}\right)^n =
\frac{1-\left(az^{-1}\right)^N}{1-az^{-1}}
=
\frac{1}{z^{N-1}}\frac{z^N-a^N}{z-a}, \quad |z|>0 \tag{53}
$$

Note that when $z=a$ the closed form is $0/0$, but the first summation reduces to a sum of $N$ ones, $X(a)=N$: the apparent pole at $z=a$ is canceled by a zero.

<a name="14.9"></a>
## 14.9 Finite- and Infinite-Duration Sequences

**Finite-duration sequence**: A sequence where values of $x[n]$ are nonzero only for a finite time interval. Its Z-transform is a finite sum, so it converges everywhere except possibly at $z=0$ (if there are terms with $n>0$) and $z=\infty$ (if there are terms with $n<0$), as Figure 14.7 shows.

In [ ]:
C = DSP_COLORS
rows = [(np.arange(0, 5), [3, 2, 1, -0.2, -2], 'causal (right-sided)', (0, np.inf), '$|z|>0$: all $z$ except the origin'),
        (np.arange(-3, 0), [2, 2.5, 2], 'anticausal (left-sided)', (-1, np.inf), r'$|z|<\infty$: all $z$, including the origin'),
        (np.arange(-3, 4), [2, 2.5, 2, 1.5, 0.8, -0.2, -2], 'two-sided', (0, np.inf), r'$0<|z|<\infty$: all $z$ except the origin')]
fig, axs = plt.subplots(3, 2, figsize=(11, 11.5), gridspec_kw=dict(width_ratios=[1.5, 1]))
for i, (nn, vals, name, roc, lab) in enumerate(rows):
    n = np.arange(-4, 6); x = np.zeros(n.size)
    x[np.searchsorted(n, nn)] = vals
    mk, st, bl = axs[i, 0].stem(n, x, basefmt='k-')
    plt.setp(st, color=C['time_domain']); plt.setp(mk, color=C['time_domain'], ms=6)
    axs[i, 0].set_xlim(-4.5, 5.5); axs[i, 0].set_ylim(-2.6, 3.4); axs[i, 0].set_xlabel('$n$'); axs[i, 0].set_ylabel('$x[n]$')
    axs[i, 0].set_title(f'({"ace"[i]}) finite, {name}', fontsize=11)
    zplane(axs[i, 1], roc=roc, lim=1.6, unit_circle=False)
    if roc[0] == 0:
        axs[i, 1].plot(0, 0, 'o', mfc='white', mec='k', ms=9, zorder=6)
    else:
        axs[i, 1].plot(0, 0, 'o', color='k', ms=6, zorder=6)
    axs[i, 1].set_title(f'({"bdf"[i]}) {lab}', fontsize=10)
plt.tight_layout(); plt.show()

<b>Figure 14.7.</b> Finite-duration sequences and their regions of convergence (shaded). (a, b) Causal: positive powers of $z^{-1}$ only, so the origin is excluded (open circle), $|z|>0$. (c, d) Anticausal: positive powers of $z$ only, so the origin is included (filled dot) and only $z=\infty$ is excluded, $|z|<\infty$. (e, f) Two-sided: both are excluded, $0<|z|<\infty$. <i>(Python-generated.)</i>

Otherwise $x[n]$ is an **infinite-duration sequence**. There are a number of different types of these infinite-duration sequences:

   * **Right-sided**: If $x[n]=0$ for $n<N_+$ where $N_+$ is an integer.
    
   * **Left-sided**: If $x[n]=0$ for $n>N_-$ where $N_-$ is an integer.
    
   * **Two-sided**: Neither a right- nor left-sided sequence.

Figure 14.8 shows one example of each, built from the sequences of Examples 2 and 3.

In [ ]:
C = DSP_COLORS
Rp, Rm = 0.6, 1.4
n = np.arange(-8, 9)
right = np.where(n >= 0, Rp**np.clip(n, 0, None), 0.0)                       # 0.6^n u[n]
left = np.where(n <= -1, Rm**np.clip(n, None, -1).astype(float), 0.0)        # 1.4^n u[-n-1]
rows = [(right, 'right-sided: $0.6^n u[n]$', (Rp, np.inf), '$|z|>R_+=0.6$'),
        (left, 'left-sided: $1.4^n u[-n-1]$', (0, Rm), '$|z|<R_-=1.4$'),
        (right + left, 'two-sided: the sum of both', (Rp, Rm), '$0.6<|z|<1.4$')]
fig, axs = plt.subplots(3, 2, figsize=(11, 11.5), gridspec_kw=dict(width_ratios=[1.5, 1]))
for i, (x, name, roc, lab) in enumerate(rows):
    mk, st, bl = axs[i, 0].stem(n, x, basefmt='k-')
    plt.setp(st, color=C['time_domain']); plt.setp(mk, color=C['time_domain'], ms=6)
    axs[i, 0].set_xlim(-9.5, 9.5); axs[i, 0].set_ylim(-0.1, 1.15); axs[i, 0].set_xlabel('$n$'); axs[i, 0].set_ylabel('$x[n]$')
    if roc[1] == np.inf or roc[0] > 0:
        axs[i, 0].text(8.7, 0.12, '…', fontsize=16, ha='center')
    if roc[1] < np.inf:
        axs[i, 0].text(-8.7, 0.12, '…', fontsize=16, ha='center')
    axs[i, 0].set_title(f'({"ace"[i]}) infinite, {name}', fontsize=11)
    zplane(axs[i, 1], roc=roc, lim=1.8)
    axs[i, 1].set_title(f'({"bdf"[i]}) ROC {lab}', fontsize=11)
plt.tight_layout(); plt.show()

<b>Figure 14.8.</b> Infinite-duration sequences and their regions of convergence (shaded; unit circle dashed). (a, b) Right-sided, $0.6^n u[n]$: ROC outside the circle $|z| = R_+ = 0.6$. (c, d) Left-sided, $1.4^n u[-n-1]$: ROC inside $|z| = R_- = 1.4$. (e, f) Two-sided (their sum): the annulus $0.6<|z|<1.4$. The ellipses mark sequences that continue beyond the plot. <i>(Python-generated.)</i>

<a name="14.10"></a>
## 14.10 Parting Thoughts

The Z-transform turns a sequence into a function of a complex variable, a delay into multiplication by
$z^{-1}$, and a difference equation into a ratio of polynomials. Everything about an LTI system is then encoded in a few numbers: its zeros, its poles,
a gain, and the region of convergence. The unit circle carries the frequency response (equation 18), and the position of the poles relative to it
decides stability (equation 38).

---
# Microquestions

<div class="alert alert-warning">
<b>Microquestion 14.1 &mdash; A wavelet, a reflectivity and an inverse filter</b>

A two-term seismic wavelet $w = [1, -0.5]$ ($w[0]$, $w[1]$) is convolved with the reflectivity $r = [1, 0, 0, -0.3]$ (a reflector at $n = 0$ and a weaker, opposite-polarity one at $n = 3$), as in the convolutional model of Module 7.

<b>(a)</b> Write $W(z)$ and $R(z)$ in powers of $z^{-1}$ and multiply them to get the recorded trace $s[n]$.

<b>(b)</b> The inverse filter is $1/W(z) = 1/(1 - 0.5z^{-1})$. Use the geometric series (MATH 213) to write its first five coefficients, state its ROC, and say whether it is stable.

<b>(c)</b> A colleague proposes the same procedure for the wavelet $w = [0.5, -1]$. Show that the causal inverse grows, and explain in terms of the zero\'s position why deconvolution of this wavelet fails (Modules 7 and 8).
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 14.1</b></summary>

<b>(a)</b> $W = 1 - 0.5z^{-1}$, $R = 1 - 0.3z^{-3}$, $S = WR = 1 - 0.5z^{-1} - 0.3z^{-3} + 0.15z^{-4}$, so $s = [1, -0.5, 0, -0.3, 0.15]$ (the same as <code>np.convolve(w, r)</code>).

<b>(b)</b> $1/(1 - 0.5z^{-1}) = \sum_n (0.5z^{-1})^n$, with coefficients $[1, 0.5, 0.25, 0.125, 0.0625, \dots]$ and ROC $|z| > 0.5$. The ROC contains the unit circle, so the causal inverse is stable (equation 38). Truncating it after six terms and convolving with $w$ leaves an error of $-0.0156$ at $n = 6$.

<b>(c)</b> $W = 0.5 - z^{-1}$ has its zero at $z = 2$. The causal inverse is $2/(1 - 2z^{-1}) = [2, 4, 8, 16, 32, \dots]$ with ROC $|z| > 2$, which excludes the unit circle: unstable. A zero outside the unit circle becomes a pole outside it in the inverse, the instability met in Section 7.9 and Module 8; a stable inverse exists only as an anticausal sequence (Distraction A).
</details>

<div class="alert alert-warning">
<b>Microquestion 14.2 &mdash; Sampling a seismometer</b>

A lightly damped seismometer obeys $\ddot{y} + 2\zeta\omega_n\dot{y} + \omega_n^2 y = 0$ with natural frequency $f_n = 1$ Hz and damping ratio $\zeta = 0.05$ (MATH 225; the PHGN 100 damped oscillator). It is recorded at 20 samples/s ($T = 0.05$ s, as for the Tohoku data of Section 12.5).

<b>(a)</b> Find the continuous roots $s$ and the discrete poles $z = e^{sT}$ (radius and angle).

<b>(b)</b> Write the LCCDE $y[n] + a_1y[n-1] + a_2y[n-2] = 0$ whose characteristic roots are those poles, and find how many samples it takes the free oscillation to decay by $1/e$.

<b>(c)</b> A student instead discretizes the ODE with forward Euler (MATH 225), which maps each root to $z = 1 + sT$. Show that the discrete system is unstable even though the seismometer is stable, and state the condition on $\omega_nT$ and $\zeta$.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 14.2</b></summary>

<b>(a)</b> $s = -\zeta\omega_n \pm i\omega_n\sqrt{1 - \zeta^2} = -0.3142 \pm 6.2753i$ s$^{-1}$. Radius $r = e^{-\zeta\omega_nT} = e^{-0.015708} = 0.98441$; angle $\theta = \omega_dT = 0.31377$ rad $= 17.98^\circ$.

<b>(b)</b> $a_1 = -2r\cos\theta = -1.87271$, $a_2 = r^2 = 0.96907$ (Section 14.3.2; an LCCDE of the kind in Module 9). The envelope $r^n$ reaches $e^{-1}$ at $n = 1/(\zeta\omega_nT) = 63.7$ samples (3.2 s). The recursion reproduces the sampled ODE solution to about $10^{-15}$.

<b>(c)</b> $|1 + sT|^2 = (1 - \zeta\omega_nT)^2 + (\omega_dT)^2 = 1 + \omega_nT(\omega_nT - 2\zeta)$, so $|1 + sT| = 1.0331 > 1$: the poles are outside the unit circle and the discrete system is unstable (equation 38). This happens whenever $\omega_nT > 2\zeta$; here $\omega_nT = 0.314 > 0.1$. The exact map $z = e^{sT}$ always sends Re $s < 0$ inside the circle.
</details>

<div class="alert alert-warning">
<b>Microquestion 14.3 &mdash; Where the boxcar is deaf</b>

The 4-point running sum $h = [1, 1, 1, 1]$ (an unnormalized boxcar smoother, Modules 7 and 12) has $H(z) = 1 + z^{-1} + z^{-2} + z^{-3}$.

<b>(a)</b> Find its zeros (hint: multiply by $(1 - z^{-1})$ and use the roots of unity, MATH 213). Evaluate $|H(e^{i\omega})|$ at $\omega = 0, \pi/4, \pi/2, 3\pi/4, \pi$ using equation 18.

<b>(b)</b> The 4-point DFT of $h$ is $[4, 0, 0, 0]$ (Module 11). Explain this using the $z$-plane (Figure 14.1), and predict the 8-point zero-padded DFT magnitudes.

<b>(c)</b> Critique: "Since its DFT is $[4, 0, 0, 0]$, the boxcar passes only zero frequency."
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 14.3</b></summary>

<b>(a)</b> $H(z)(1 - z^{-1}) = 1 - z^{-4}$, whose roots are the four 4th roots of unity; removing $z = 1$ leaves the zeros $z = -1$ and $z = \pm i$. Then $|H| = 4, 2.613, 0, 1.082, 0$ at the five frequencies ($= |\sin 2\omega / \sin(\omega/2)|$).

<b>(b)</b> The 4-point DFT samples the unit circle at $1, i, -1, -i$, and three of these points are zeros. Zero padding to 8 adds the points $e^{\pm i\pi/4}$ and $e^{\pm i3\pi/4}$: $|X| = [4, 2.613, 0, 1.082, 0, 1.082, 0, 2.613]$.

<b>(c)</b> False: between the zeros $|H|$ is large (2.61 at $\pi/4$, 1.08 at $3\pi/4$: the sidelobes of the boxcar window of Module 12). A DFT whose length equals the filter length happens to sample exactly at the zeros, a picket-fence effect (Section 11.12).
</details>